# Lab 6: pooling i pełna CNN, FashionMNIST (kod)
Ten notebook zawiera **tylko kod i eksperymenty**. Sprawozdanie jest w osobnym pliku `REPORT.md`; notebook zapisuje wyniki do `figures/` i `results/`.

**Uruchom na GPU:** Runtime → Change runtime type → GPU, potem Runtime → Run all (ok. 15–20 minut: 9 konfiguracji × 3 seedy × 10 epok).

Różnice względem notebooka z ćwiczenia: wybór epoki i modelu robimy na **walidacji** (a nie po najlepszym wyniku na zbiorze testowym), każdy wariant trenujemy na **3 seedach** (różnice między wariantami są małe) i zbiór testowy używamy **raz**, na końcu.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

EPOCHS = 10  # ten sam budżet epok dla wszystkich wariantów
BATCH = 128
LR = 1e-3  # Adam, ten sam dla wszystkich wariantów
SEEDS = (0, 1, 2)
device = "cuda" if torch.cuda.is_available() else "cpu"
hardware = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"

FIG_DIR, RES_DIR = Path("figures"), Path("results")
for d in (FIG_DIR, RES_DIR):
    d.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "| sprzęt:", hardware)


def sync():
    # poprawny pomiar czasu na GPU wymaga synchronizacji
    if device == "cuda":
        torch.cuda.synchronize()


def warmup_gpu():
    if device != "cuda":
        return
    m = nn.Sequential(
        nn.Conv2d(1, 8, 3),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(8, 10),
    ).to(device)
    m(torch.randn(64, 1, 28, 28, device=device)).sum().backward()
    sync()


warmup_gpu()

## 1. Dane: FashionMNIST i stały podział 50 000 / 10 000 / 10 000
Zbiór treningowy (60 000) dzielimy **raz** na 50 000 treningowych i 10 000 walidacyjnych (stała permutacja, seed 42). Oficjalny zbiór testowy (10 000) nie bierze udziału w żadnych decyzjach. Dane trzymamy w pamięci GPU (czas uczenia mierzy obliczenia).

In [ ]:
mean, std = 0.2860, 0.3530


def load(train):
    ds = torchvision.datasets.FashionMNIST(
        "./data", train=train, download=True
    )
    x = ((ds.data.float() / 255.0 - mean) / std).unsqueeze(1)  # [N, 1, 28, 28]
    return x, ds.targets.clone(), ds.classes


x_all, y_all, classes = load(True)
x_te, y_te, _ = load(False)

split_gen = torch.Generator().manual_seed(42)
perm = torch.randperm(len(x_all), generator=split_gen)
train_idx, val_idx = perm[:50_000], perm[50_000:]
assert set(train_idx.tolist()).isdisjoint(val_idx.tolist())
np.savez(
    RES_DIR / "split_idx.npz", train=train_idx.numpy(), val=val_idx.numpy()
)

x_train, y_train = x_all[train_idx].to(device), y_all[train_idx].to(device)
x_val, y_val = x_all[val_idx].to(device), y_all[val_idx].to(device)
x_test, y_test = x_te.to(device), y_te.to(device)
print("Klasy:", classes)
print(
    f"Trening: {len(x_train)} | walidacja: {len(x_val)} | test: {len(x_test)}"
)
print("Kształt wejścia:", tuple(x_train.shape[1:]))

## 2. Mechanika poolingu i rozmiar wyjścia
Sprawdzamy na macierzy ze wstępu (max → `[[6, 8], [14, 16]]`, average → `[[4, 5], [11.5, 13.5]]`, GAP → 8,5) oraz wzór `out = ⌊(L − k) / s⌋ + 1` względem prawdziwych warstw.

In [ ]:
x4 = torch.tensor(
    [
        [2.0, 3.0, 1.0, 4.0],
        [5.0, 6.0, 7.0, 8.0],
        [9.0, 10.0, 11.0, 12.0],
        [13.0, 14.0, 15.0, 16.0],
    ]
).view(1, 1, 4, 4)
max_map = nn.MaxPool2d(2)(x4)[0, 0]
avg_map = nn.AvgPool2d(2)(x4)[0, 0]
gap = nn.AdaptiveAvgPool2d(1)(x4)
assert max_map.tolist() == [[6.0, 8.0], [14.0, 16.0]]
assert avg_map.tolist() == [[4.0, 5.0], [11.5, 13.5]]
print("max 2x2:", max_map.tolist())
print("avg 2x2:", avg_map.tolist())
print("GAP:", gap.item(), "| kształt", tuple(gap.shape))


def pool_out(length, k, s):
    return (length - k) // s + 1


rows = []
for length, k, s in [
    (28, 2, 2),
    (14, 2, 2),
    (7, 2, 2),
    (28, 3, 3),
    (9, 3, 3),
    (28, 3, 2),
    (13, 3, 2),
]:
    real = nn.MaxPool2d(k, s)(torch.zeros(1, 1, length, length)).shape[-1]
    rows.append(
        {
            "wejście L": length,
            "okno k": k,
            "krok s": s,
            "wzór": pool_out(length, k, s),
            "MaxPool2d": real,
            "zgodne": "tak" if real == pool_out(length, k, s) else "NIE",
        }
    )
pool_dims = pd.DataFrame(rows)
pool_dims.to_csv(RES_DIR / "pool_dims.csv", index=False)
display(pool_dims)
assert (pool_dims["zgodne"] == "tak").all()

In [ ]:
img = x_val[:1].cpu()
line_bw = torch.ones(1, 1, 16, 16)  # czarna kreska na białym tle
for i in range(16):
    line_bw[0, 0, i, 15 - i] = 0.0
line_wb = 1.0 - line_bw  # biała kreska na czarnym tle

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax in axes.flat:
    ax.axis("off")
panels_top = [
    ("oryginał 28x28", img),
    ("max 2x2 (14x14)", nn.MaxPool2d(2)(img)),
    ("avg 2x2 (14x14)", nn.AvgPool2d(2)(img)),
    ("max 3x3 (9x9)", nn.MaxPool2d(3)(img)),
    ("avg 3x3 (9x9)", nn.AvgPool2d(3)(img)),
]
for ax, (title, t) in zip(axes[0], panels_top):
    ax.imshow(t[0, 0], cmap="gray", interpolation="nearest")
    ax.set_title(title, fontsize=9)
panels_bottom = [
    ("czarna kreska", line_bw),
    ("max 2x2", nn.MaxPool2d(2)(line_bw)),
    ("avg 2x2", nn.AvgPool2d(2)(line_bw)),
    ("biała kreska", line_wb),
    ("max 2x2", nn.MaxPool2d(2)(line_wb)),
    ("avg 2x2", nn.AvgPool2d(2)(line_wb)),
]
for ax, (title, t) in zip(axes[1], panels_bottom):
    ax.imshow(t[0, 0], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "pooling_demo.png", dpi=150)
plt.show()

## 3. Model: pełna CNN z konfigurowalną redukcją i głowicą
Blok: `Conv3×3 (padding 1) → BatchNorm → ReLU → redukcja`, redukcja = max pooling, average pooling albo dodatkowy `Conv3×3 (stride 2, padding 1) → BatchNorm → ReLU`. Głowica: GAP → Flatten → Linear (lub, w eksperymencie dodatkowym, Flatten → Linear).

In [ ]:
class CNN(nn.Module):
    def __init__(
        self,
        reduce="max",
        channels=(32, 64),
        window=2,
        pool_stride=None,
        head="gap",
        n_classes=10,
        in_ch=1,
        in_size=28,
    ):
        super().__init__()

        def block(cin, cout):
            layers = [
                nn.Conv2d(cin, cout, 3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(),
            ]
            if reduce == "max":
                layers.append(nn.MaxPool2d(window, pool_stride))
            elif reduce == "avg":
                layers.append(nn.AvgPool2d(window, pool_stride))
            elif reduce == "stride":
                layers += [
                    nn.Conv2d(cout, cout, 3, stride=2, padding=1),
                    nn.BatchNorm2d(cout),
                    nn.ReLU(),
                ]
            else:
                raise ValueError(reduce)
            return nn.Sequential(*layers)

        blocks, cin = [], in_ch
        for cout in channels:
            blocks.append(block(cin, cout))
            cin = cout
        self.features = nn.Sequential(*blocks)
        self.features.eval()  # pomiar kształtu nie może zmieniać statystyk BN
        with torch.no_grad():  # kształt mapy cech przed głowicą
            shape = self.features(torch.zeros(1, in_ch, in_size, in_size))
        self.features.train()
        self.feat_shape = tuple(shape.shape[1:])
        if head == "gap":
            self.head = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(self.feat_shape[0], n_classes),
            )
        else:
            n_flat = int(np.prod(self.feat_shape))
            self.head = nn.Sequential(
                nn.Flatten(), nn.Linear(n_flat, n_classes)
            )

    def forward(self, x):
        return self.head(self.features(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters())


for name in ("max", "avg", "stride"):
    m = CNN(reduce=name).eval()
    with torch.no_grad():
        out = m(torch.zeros(2, 1, 28, 28))
    print(
        f"{name:7s} | mapa przed głowicą {m.feat_shape} | "
        f"parametry {count_params(m):,} | logity {tuple(out.shape)}"
    )

## 4. Trening z wyborem epoki na walidacji
Jedna funkcja `run` dla wszystkich wariantów. Po każdej epoce liczymy stratę i accuracy na walidacji; zapamiętujemy checkpoint z **najniższą stratą walidacyjną**. Czas mierzymy tylko dla pętli treningowej (z synchronizacją GPU).

In [ ]:
def batches(x, y, batch_size, gen=None):
    n = len(x)
    if gen is None:
        idx = torch.arange(n, device=x.device)
    else:
        idx = torch.randperm(n, generator=gen).to(x.device)
    for i in range(0, n, batch_size):
        j = idx[i : i + batch_size]
        yield x[j], y[j]


@torch.no_grad()
def evaluate(model, x, y, batch_size=1000):
    model.eval()
    crit = nn.CrossEntropyLoss(reduction="sum")
    loss_sum, correct = 0.0, 0
    for xb, yb in batches(x, y, batch_size):
        out = model(xb)
        loss_sum += crit(out, yb).item()
        correct += (out.argmax(1) == yb).sum().item()
    return loss_sum / len(x), correct / len(x)


def clone_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def run(name, factory, seed, epochs=EPOCHS, lr=LR):
    torch.manual_seed(seed)
    model = factory().to(device)
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    keys = ["train_loss", "val_loss", "val_acc", "epoch_time"]
    hist = {k: [] for k in keys}
    best_val, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = torch.zeros((), device=device)
        sync()
        t0 = time.perf_counter()
        for xb, yb in batches(x_train, y_train, BATCH, gen):
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            loss_sum += loss.detach() * yb.numel()
        sync()
        hist["epoch_time"].append(time.perf_counter() - t0)
        hist["train_loss"].append(loss_sum.item() / len(x_train))
        val_loss, val_acc = evaluate(model, x_val, y_val)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        if val_loss < best_val:  # wybór epoki po stracie walidacyjnej
            best_val, best_epoch = val_loss, epoch
            best_state = clone_state(model)
    return {
        "name": name,
        "seed": seed,
        "factory": factory,
        "params": count_params(model),
        "feat_shape": model.feat_shape,
        "hist": hist,
        "best_epoch": best_epoch,
        "best_state": best_state,
        "train_time": sum(hist["epoch_time"]),
    }


def run_all(configs, seeds=SEEDS):
    out = {}
    for name, factory in configs.items():
        runs = []
        for seed in seeds:
            r = run(name, factory, seed)
            i = r["best_epoch"] - 1
            print(
                f"{name:38s} seed {seed} | epoka {r['best_epoch']:2d} | "
                f"val acc {r['hist']['val_acc'][i] * 100:.2f}% | "
                f"{r['train_time']:.0f} s"
            )
            runs.append(r)
        out[name] = runs
    return out


def fmt(vals, digits=2):
    return f"{np.mean(vals):.{digits}f} ± {np.std(vals, ddof=1):.{digits}f}"


def val_acc_of(r):
    return r["hist"]["val_acc"][r["best_epoch"] - 1] * 100


def summarize(name, runs):
    c, h, w = runs[0]["feat_shape"]
    return {
        "wariant": name,
        "parametry": runs[0]["params"],
        "mapa przed głowicą": f"{c}x{h}x{w}",
        "val acc [%] (średnia ± std)": fmt([val_acc_of(r) for r in runs]),
        "wybrane epoki": ",".join(str(r["best_epoch"]) for r in runs),
        "czas uczenia [s]": round(np.mean([r["train_time"] for r in runs]), 1),
    }

## 5. Główne porównanie: max pooling vs average pooling vs splot z krokiem 2
Ten sam zbiór i podział, optymalizator (Adam, lr = 1e-3), batch 128, **10 epok**, 3 seedy na wariant.

In [ ]:
main_cfg = {
    "max pooling 2x2": lambda: CNN(reduce="max"),
    "average pooling 2x2": lambda: CNN(reduce="avg"),
    "splot stride=2": lambda: CNN(reduce="stride"),
}
main_runs = run_all(main_cfg)

In [ ]:
colors = {
    "max pooling 2x2": "tab:blue",
    "average pooling 2x2": "tab:green",
    "splot stride=2": "tab:red",
}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ep = np.arange(1, EPOCHS + 1)
for name, runs in main_runs.items():
    acc = np.array([r["hist"]["val_acc"] for r in runs]) * 100
    loss = np.array([r["hist"]["val_loss"] for r in runs])
    c = colors[name]
    ax1.plot(ep, acc.mean(0), "-o", color=c, ms=3, label=name)
    ax1.fill_between(ep, acc.min(0), acc.max(0), color=c, alpha=0.15)
    ax2.plot(ep, loss.mean(0), "-o", color=c, ms=3, label=name)
    ax2.fill_between(ep, loss.min(0), loss.max(0), color=c, alpha=0.15)
ax1.set_xlabel("epoka")
ax1.set_ylabel("accuracy walidacyjna [%]")
ax1.set_title("Accuracy walidacyjna (średnia, pasmo min-max z 3 seedów)")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
ax2.set_xlabel("epoka")
ax2.set_ylabel("strata walidacyjna")
ax2.set_title("Strata walidacyjna")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "curves_pooling_variants.png", dpi=150)
plt.show()

## 6. Końcowa ocena na zbiorze testowym (decyzja zamrożona)
**Uwaga:** to jedyne miejsce, w którym używamy zbioru testowego, i tylko dla głównych wariantów. Oceniamy checkpointy wybrane na walidacji (najniższa strata walidacyjna) dla każdego seeda.

In [ ]:
rows = []
for name, runs in main_runs.items():
    test_accs = []
    for r in runs:
        model = r["factory"]().to(device)
        model.load_state_dict(r["best_state"])
        r["test_acc"] = evaluate(model, x_test, y_test)[1] * 100
        test_accs.append(r["test_acc"])
    s = summarize(name, runs)
    s["test acc [%] (średnia ± std)"] = fmt(test_accs)
    rows.append(s)
main_df = pd.DataFrame(rows)
main_df.to_csv(RES_DIR / "main_comparison.csv", index=False)
display(main_df)
print(main_df.to_markdown(index=False))

per_run = pd.DataFrame(
    [
        {
            "wariant": r["name"],
            "seed": r["seed"],
            "wybrana epoka": r["best_epoch"],
            "val acc [%]": round(val_acc_of(r), 2),
            "val loss": round(r["hist"]["val_loss"][r["best_epoch"] - 1], 4),
            "test acc [%]": round(r["test_acc"], 2),
            "czas uczenia [s]": round(r["train_time"], 1),
        }
        for runs in main_runs.values()
        for r in runs
    ]
)
per_run.to_csv(RES_DIR / "per_run_main.csv", index=False)
display(per_run)

config = {
    "zbior": "FashionMNIST",
    "podzial": "50000 / 10000 / 10000 (seed 42)",
    "optymalizator": "Adam",
    "lr": LR,
    "batch": BATCH,
    "epoki": EPOCHS,
    "seedy": list(SEEDS),
    "wybor_checkpointu": "najnizsza strata walidacyjna",
    "sprzet": hardware,
    "pytorch": torch.__version__,
}
json.dump(config, open(RES_DIR / "run_config.json", "w"), indent=2)
print(json.dumps(config, indent=2, ensure_ascii=False))

## 7. Zakres dodatkowy: okno poolingu, głowica GAP vs Flatten, liczba bloków
Wszystkie warianty dodatkowe oceniamy na **walidacji** (3 seedy, 10 epok); bazą jest max pooling 2×2 z GAP z głównego porównania.

In [ ]:
extra_cfg = {
    "max 3x3, krok 3": lambda: CNN(reduce="max", window=3),
    "max 3x3, krok 2 (nakładające się)": lambda: CNN(
        reduce="max", window=3, pool_stride=2
    ),
    "max 2x2, Flatten + Linear (bez GAP)": lambda: CNN(
        reduce="max", head="flatten"
    ),
    "1 blok (32)": lambda: CNN(reduce="max", channels=(32,)),
    "3 bloki (32-64-128)": lambda: CNN(reduce="max", channels=(32, 64, 128)),
    "4 bloki (32-64-128-256)": lambda: CNN(
        reduce="max", channels=(32, 64, 128, 256)
    ),
}
extra_runs = run_all(extra_cfg)

In [ ]:
base = "max pooling 2x2"
groups = [
    (
        "okno poolingu",
        [base, "max 3x3, krok 3", "max 3x3, krok 2 (nakładające się)"],
    ),
    ("głowica", [base, "max 2x2, Flatten + Linear (bez GAP)"]),
    (
        "liczba bloków",
        [
            "1 blok (32)",
            base,
            "3 bloki (32-64-128)",
            "4 bloki (32-64-128-256)",
        ],
    ),
]
all_runs = {**main_runs, **extra_runs}
rows = []
for group, names in groups:
    for n in names:
        s = summarize(n, all_runs[n])
        s["eksperyment"] = group
        rows.append(s)
extra_df = pd.DataFrame(rows)[
    [
        "eksperyment",
        "wariant",
        "parametry",
        "mapa przed głowicą",
        "val acc [%] (średnia ± std)",
        "wybrane epoki",
        "czas uczenia [s]",
    ]
]
extra_df.to_csv(RES_DIR / "extras.csv", index=False)
display(extra_df)
print(extra_df.to_markdown(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (group, names) in zip(axes, groups):
    means = [np.mean([val_acc_of(r) for r in all_runs[n]]) for n in names]
    stds = [
        np.std([val_acc_of(r) for r in all_runs[n]], ddof=1) for n in names
    ]
    labels = [
        f"{n.replace(' (nakładające się)', '').replace(' (bez GAP)', '')}\n"
        f"{all_runs[n][0]['params']:,} par."
        for n in names
    ]
    ax.bar(
        range(len(names)),
        means,
        yerr=stds,
        capsize=4,
        color="tab:blue",
        alpha=0.8,
    )
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(labels, fontsize=7)
    lo = min(means) - 3 * max(stds) - 0.3
    ax.set_ylim(lo, max(means) + 3 * max(stds) + 0.3)
    ax.set_ylabel("val accuracy [%]")
    ax.set_title(group)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "extras_overview.png", dpi=150)
plt.show()

## 8. Pliki wynikowe
Do sprawozdania trafiają: `figures/*.png` oraz `results/*.csv|json`. Kod i sprawozdanie są osobnymi plikami.